# 1. Audio Download & Speech Classification

Three independent, resumable stages, each checkpointed to its own CSV so a failure in one doesn't force redoing the others:

1. **Export** — query the DB once into `data/recordings.csv` (cached; reruns reuse it unless `FORCE_REFRESH_DB=1`).
2. **Download** — read `recordings.csv`, HEAD/range-check each `streamableUrl`, then stream the raw bytes straight to `data/audio/{id}.{ext}` (gitignored) with `requests`. No decoding, no `ffmpeg` subprocess per file, so this scales to thousands of files without being bottlenecked on subprocess-spawn overhead. Writes to a `.partial` temp file first and only renames it into place on success, so an interrupted download never masquerades as a completed one on the next run.
3. **Classify** — read `recordings.csv`, and for whatever's been downloaded, decode it with a single local `ffmpeg` call (no network involved, so it's fast) straight into an in-memory PCM waveform, run Silero VAD on it, and append the original DB columns plus `is_speech` (boolean) to `data/audio_speech_labels.csv`. Rows that fail or have no downloaded audio are skipped and logged, not written.

**Why decoding happens in Stage 3, not Stage 2:** downloading raw bytes keeps Stage 2 to pure I/O — fast and cheap to retry. The transcode cost isn't eliminated, just deferred to classify time, where it runs against a local file instead of a network stream. Requires `ffmpeg` on `PATH`.

In [1]:
import os
import subprocess
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from urllib.parse import urlparse

import numpy as np
import pandas as pd
import psycopg
import requests
import torch
from dotenv import load_dotenv
from silero_vad import get_speech_timestamps, load_silero_vad
from tqdm import tqdm

load_dotenv()

True

## Configuration

All values come from `.env` at the repo root (found automatically by `load_dotenv()` walking up from this notebook's directory), with sensible fallbacks.

In [2]:
# Database
DB_HOST = os.getenv("DB_HOST")
DB_PORT = int(os.getenv("DB_PORT", "5432"))
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
TABLE_NAME = os.getenv("TABLE_NAME")

# Name of the column containing the streamable audio URL.
AUDIO_URL_COLUMN = os.getenv("AUDIO_URL_COLUMN", "streamableUrl")

# Name of the row identifier column, used for resuming and for naming
# downloaded audio files on disk.
ID_COLUMN = os.getenv("ID_COLUMN", "id")

_where_clause_env = os.getenv("WHERE_CLAUSE", "").strip()
WHERE_CLAUSE = _where_clause_env if _where_clause_env else None

_limit_env = os.getenv("LIMIT", "").strip()
LIMIT = int(_limit_env) if _limit_env else None

# Re-query the DB even if a cached recordings.csv already exists.
FORCE_REFRESH_DB = os.getenv("FORCE_REFRESH_DB", "").strip().lower() in {"1", "true", "yes"}

# Output locations (relative to this notebook's directory, i.e. jupyter_notebooks/)
DATA_DIR = Path(os.getenv("DATA_DIR", "../data"))
AUDIO_DIR = DATA_DIR / "audio"
RECORDINGS_CSV = DATA_DIR / "recordings.csv"
OUTPUT_CSV = DATA_DIR / os.getenv("OUTPUT_CSV", "audio_speech_labels.csv")
DOWNLOAD_LOG_CSV = DATA_DIR / os.getenv("DOWNLOAD_LOG_CSV", "audio_download_log.csv")
SAVE_EVERY = int(os.getenv("SAVE_EVERY", "10"))

DATA_DIR.mkdir(parents=True, exist_ok=True)
AUDIO_DIR.mkdir(parents=True, exist_ok=True)

# Speech detection
SAMPLE_RATE = int(os.getenv("SAMPLE_RATE", "16000"))

# A recording is labelled as speech when at least this proportion of it
# contains detected speech.
SPEECH_RATIO_THRESHOLD = float(os.getenv("SPEECH_RATIO_THRESHOLD", "0.20"))

MIN_SPEECH_DURATION_MS = int(os.getenv("MIN_SPEECH_DURATION_MS", "250"))
MIN_SILENCE_DURATION_MS = int(os.getenv("MIN_SILENCE_DURATION_MS", "300"))
SPEECH_PAD_MS = int(os.getenv("SPEECH_PAD_MS", "100"))

# Stage 2 (raw HTTP download) settings
DOWNLOAD_TIMEOUT_SECONDS = int(os.getenv("DOWNLOAD_TIMEOUT_SECONDS", "60"))
MIN_FILE_BYTES = int(os.getenv("MIN_FILE_BYTES", "1024"))

# Number of parallel workers. Stage 2 uses these for HTTP downloads; Stage 3
# uses these for ffmpeg-decode + VAD, each thread getting its own Silero
# model instance.
MAX_WORKERS = int(os.getenv("MAX_WORKERS", "8"))

torch.set_num_threads(1)

## Stage 1 — Export DB rows to `recordings.csv`

If `recordings.csv` already exists, it's reused as-is — no need to hit the DB twice.

In [3]:
def connect_to_database():
    return psycopg.connect(
        host=DB_HOST,
        port=DB_PORT,
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD,
    )


def load_database_rows() -> pd.DataFrame:
    query = f"SELECT * FROM {TABLE_NAME}"

    if WHERE_CLAUSE:
        query += f" WHERE {WHERE_CLAUSE}"

    if LIMIT is not None:
        query += f" LIMIT {int(LIMIT)}"

    print("Reading rows from PostgreSQL...")

    with connect_to_database() as connection:
        with connection.cursor() as cursor:
            cursor.execute(query)
            column_names = [description.name for description in cursor.description]
            rows = cursor.fetchall()

    dataframe = pd.DataFrame(rows, columns=column_names)

    print(f"Loaded {len(dataframe)} rows.")

    return dataframe

In [4]:
if RECORDINGS_CSV.exists() and not FORCE_REFRESH_DB:
    recordings = pd.read_csv(RECORDINGS_CSV)
    print(
        f"Loaded {len(recordings)} cached rows from {RECORDINGS_CSV}.\n"
        "Set FORCE_REFRESH_DB=1 in .env to re-query the DB instead."
    )
else:
    recordings = load_database_rows()

    if AUDIO_URL_COLUMN not in recordings.columns:
        raise ValueError(
            f"Column '{AUDIO_URL_COLUMN}' was not found.\n"
            f"Available columns: {list(recordings.columns)}"
        )

    if ID_COLUMN not in recordings.columns:
        raise ValueError(
            f"Column '{ID_COLUMN}' was not found.\n"
            f"Available columns: {list(recordings.columns)}"
        )

    recordings.to_csv(RECORDINGS_CSV, index=False)
    print(f"Saved {len(recordings)} rows to {RECORDINGS_CSV}")

Loaded 6461 cached rows from ../data/recordings.csv.
Set FORCE_REFRESH_DB=1 in .env to re-query the DB instead.


## Stage 2 — Download audio (raw bytes, no transcode)

Adapted from the reference `download_streamable_audio.py`: check the URL is reachable (HEAD, falling back to a ranged GET for S3 buckets that reject HEAD), then stream the response body straight to disk under its original extension. No `ffmpeg`, no re-encoding — just I/O, which is why this scales to thousands of files much better than spawning one ffmpeg process per download.

In [5]:
def cache_path_for(row_id: str, url: str) -> Path:
    suffix = Path(urlparse(url).path).suffix.lower() or ".mp3"
    if suffix not in {".mp3", ".m4a", ".wav", ".ogg", ".aac", ".flac"}:
        suffix = ".mp3"
    return AUDIO_DIR / f"{row_id}{suffix}"


def is_cached(path: Path) -> bool:
    return path.exists() and path.stat().st_size >= MIN_FILE_BYTES


def check_streamable_url(url: str, timeout: int = DOWNLOAD_TIMEOUT_SECONDS) -> None:
    """Raise if the streamable URL is not reachable."""
    try:
        response = requests.head(url, timeout=timeout, allow_redirects=True)
        # Some S3 buckets reject HEAD; fall through to a ranged GET.
        if response.status_code in {403, 405}:
            response = requests.get(
                url,
                timeout=timeout,
                stream=True,
                headers={"Range": "bytes=0-0"},
            )
    except requests.RequestException as error:
        raise RuntimeError(f"URL check failed: {error}") from error

    if response.status_code >= 400:
        raise RuntimeError(f"URL not reachable (HTTP {response.status_code}).")

    response.close()


def download_audio(url: str, destination: Path) -> int:
    """Stream `url` to `destination` unmodified. Returns the file size in bytes."""
    temporary_path = destination.with_suffix(destination.suffix + ".partial")

    try:
        with requests.get(url, timeout=DOWNLOAD_TIMEOUT_SECONDS, stream=True) as response:
            if response.status_code >= 400:
                raise RuntimeError(f"Download failed (HTTP {response.status_code}).")

            with open(temporary_path, "wb") as handle:
                for chunk in response.iter_content(chunk_size=64 * 1024):
                    if chunk:
                        handle.write(chunk)

        size = temporary_path.stat().st_size
        if size < MIN_FILE_BYTES:
            raise RuntimeError(f"Downloaded file too small ({size} bytes).")

        temporary_path.replace(destination)
        return size

    finally:
        if temporary_path.exists():
            temporary_path.unlink(missing_ok=True)


def download_row(row: dict) -> dict:
    """Download one row's audio. Returns a log entry dict."""
    row_id = str(row.get(ID_COLUMN))
    url_value = row.get(AUDIO_URL_COLUMN)

    result = {
        "id": row_id,
        "streamable_url": None,
        "local_path": None,
        "bytes": None,
        "download_status": None,
        "download_error": None,
    }

    try:
        if pd.isna(url_value) or not str(url_value).strip():
            raise ValueError("Streamable URL is missing.")

        url = str(url_value).strip()
        result["streamable_url"] = url

        destination = cache_path_for(row_id, url)
        result["local_path"] = str(destination)

        if is_cached(destination):
            result["bytes"] = destination.stat().st_size
            result["download_status"] = "skipped_cached"
            return result

        check_streamable_url(url)
        size = download_audio(url, destination)
        result["bytes"] = size
        result["download_status"] = "downloaded"

    except Exception as error:
        result["download_status"] = "failed"
        result["download_error"] = str(error)

    return result

In [6]:
recordings = pd.read_csv(RECORDINGS_CSV)

print(f"Rows to check: {len(recordings)}")
print(f"Workers:       {MAX_WORKERS}")
print(f"Cache dir:     {AUDIO_DIR.resolve()}")

download_results = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [
        executor.submit(download_row, row.to_dict())
        for _, row in recordings.iterrows()
    ]

    for future in tqdm(as_completed(futures), total=len(futures), desc="Downloading audio", unit="file"):
        download_results.append(future.result())

download_log = pd.DataFrame(download_results)
download_log.to_csv(DOWNLOAD_LOG_CSV, index=False)

status_counts = download_log["download_status"].value_counts().to_dict()
print("\nDownload stage finished.")
print(f"Status counts: {status_counts}")
print(f"Log saved to:  {DOWNLOAD_LOG_CSV}")

Rows to check: 6461
Workers:       8
Cache dir:     /Users/tavishikaushik/Downloads/SaysUserContentRecommendation/data/audio



Download stage finished.
Status counts: {'downloaded': 6371, 'failed': 90}
Log saved to:  ../data/audio_download_log.csv


## Stage 3 — Classify audio → `audio_speech_labels.csv`

Since Stage 2 no longer transcodes, each file here is decoded on demand with one local `ffmpeg` call (no network — just reading a file already on disk — so this is fast) straight into an in-memory mono 16kHz PCM waveform, which is what Silero VAD expects.

In [7]:
_thread_local = threading.local()


def get_vad_model():
    if not hasattr(_thread_local, "model"):
        _thread_local.model = load_silero_vad()
    return _thread_local.model


def decode_audio_to_waveform(audio_path: Path, sample_rate: int = SAMPLE_RATE) -> torch.Tensor:
    """Decode any ffmpeg-readable audio file into a mono float32 waveform tensor."""
    command = [
        "ffmpeg",
        "-loglevel", "error",
        "-i", str(audio_path),
        "-vn",
        "-ac", "1",
        "-ar", str(sample_rate),
        "-f", "s16le",
        "-",
    ]

    result = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE)

    if result.returncode != 0:
        error_message = result.stderr.decode("utf-8", errors="ignore").strip()
        raise RuntimeError(error_message or "ffmpeg failed to decode the audio.")

    samples = np.frombuffer(result.stdout, dtype=np.int16).astype(np.float32) / 32768.0
    return torch.from_numpy(samples)


def classify_audio(audio_path: Path) -> bool:
    """Return True if the audio at `audio_path` is majority speech."""
    waveform = decode_audio_to_waveform(audio_path)

    total_duration_seconds = waveform.numel() / SAMPLE_RATE

    speech_segments = get_speech_timestamps(
        waveform,
        get_vad_model(),
        sampling_rate=SAMPLE_RATE,
        return_seconds=True,
        min_speech_duration_ms=MIN_SPEECH_DURATION_MS,
        min_silence_duration_ms=MIN_SILENCE_DURATION_MS,
        speech_pad_ms=SPEECH_PAD_MS,
    )

    speech_duration_seconds = sum(
        max(0.0, float(segment["end"]) - float(segment["start"]))
        for segment in speech_segments
    )

    if total_duration_seconds > 0:
        speech_ratio = speech_duration_seconds / total_duration_seconds
    else:
        speech_ratio = 0.0

    return speech_ratio >= SPEECH_RATIO_THRESHOLD


def classify_row(row: dict):
    """Classify one row's downloaded audio. Returns the DB row + is_speech, or None to skip it."""
    row_id = row.get(ID_COLUMN)
    url_value = row.get(AUDIO_URL_COLUMN)

    if pd.isna(url_value) or not str(url_value).strip():
        print(f"[CLASSIFY SKIPPED] {ID_COLUMN}={row_id}: no streamable URL.")
        return None

    audio_path = cache_path_for(str(row_id), str(url_value).strip())

    if not audio_path.exists() or audio_path.stat().st_size == 0:
        print(f"[CLASSIFY SKIPPED] {ID_COLUMN}={row_id}: audio not downloaded.")
        return None

    try:
        result = dict(row)
        result["is_speech"] = classify_audio(audio_path)
        return result

    except Exception as error:
        print(f"[CLASSIFY FAILED] {ID_COLUMN}={row_id}: {error}")
        return None

In [8]:
def load_existing_results() -> pd.DataFrame:
    if not OUTPUT_CSV.exists():
        return pd.DataFrame()

    try:
        return pd.read_csv(OUTPUT_CSV)
    except Exception as error:
        print(f"Could not read existing CSV: {error}")
        return pd.DataFrame()


def get_processed_ids(existing_results: pd.DataFrame) -> set:
    if existing_results.empty or ID_COLUMN not in existing_results.columns:
        return set()
    return set(existing_results[ID_COLUMN].astype(str).tolist())


def save_results(existing_results: pd.DataFrame, new_results: list) -> pd.DataFrame:
    new_dataframe = pd.DataFrame(new_results)

    if existing_results.empty:
        combined_dataframe = new_dataframe
    else:
        combined_dataframe = pd.concat([existing_results, new_dataframe], ignore_index=True)

    combined_dataframe.to_csv(OUTPUT_CSV, index=False)
    return combined_dataframe

In [9]:
recordings = pd.read_csv(RECORDINGS_CSV)

existing_results = load_existing_results()
processed_ids = get_processed_ids(existing_results)

rows_to_process = recordings[~recordings[ID_COLUMN].astype(str).isin(processed_ids)]

print(f"Already processed: {len(processed_ids)}")
print(f"Remaining rows:    {len(rows_to_process)}")
print(f"Workers:           {MAX_WORKERS}")

new_results = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [
        executor.submit(classify_row, row.to_dict())
        for _, row in rows_to_process.iterrows()
    ]

    for future in tqdm(as_completed(futures), total=len(futures), desc="Classifying audio", unit="file"):
        result = future.result()

        if result is not None:
            new_results.append(result)

        if new_results and len(new_results) % SAVE_EVERY == 0:
            existing_results = save_results(existing_results, new_results)
            new_results = []

if new_results:
    existing_results = save_results(existing_results, new_results)

print("\nClassification stage finished.")
print(f"Results saved to: {OUTPUT_CSV}")

Already processed: 6374
Remaining rows:    87
Workers:           8
[CLASSIFY SKIPPED] id=0e4abb65-e025-42b3-9c43-6ac2e18a462a: audio not downloaded.
[CLASSIFY SKIPPED] id=4c29a182-ed6b-413a-aaf1-d3c0a7780ad6: audio not downloaded.
[CLASSIFY SKIPPED] id=1a79d82d-1c18-4549-a157-4bd6be7fc390: audio not downloaded.
[CLASSIFY SKIPPED] id=e67ab03e-ddd8-4b64-b81f-5d1afd1fc67e: audio not downloaded.
[CLASSIFY SKIPPED] id=3c940314-9e9d-4cef-b682-86a2a5d39af9: no streamable URL.
[CLASSIFY SKIPPED] id=3f0e3e11-cf89-4255-a03e-267a972fabd7: no streamable URL.
[CLASSIFY SKIPPED] id=fb0138c1-9f3f-4faf-9643-960706fe6f96: no streamable URL.
[CLASSIFY SKIPPED] id=607ec7d9-ea87-4413-9a0e-83898339862c: audio not downloaded.
[CLASSIFY SKIPPED] id=8d5d4428-bbf5-4657-9f8e-8517b765a657: no streamable URL.
[CLASSIFY SKIPPED] id=c4ce8643-2af2-4f44-a25f-830813df575a: no streamable URL.
[CLASSIFY SKIPPED] id=e4d51c1a-1795-4acb-a092-9b14fd43e151: no streamable URL.
[CLASSIFY SKIPPED] id=89114e71-7238-4971-9ef1-15a

Classifying audio: 100%|██████████| 87/87 [00:00<00:00, 376578.38file/s]

[CLASSIFY SKIPPED] id=3f53999d-f72b-4d1b-bf1b-2678731d40cc: audio not downloaded.

Classification stage finished.
Results saved to: ../data/audio_speech_labels.csv
